# CivicLoop Step 5.3: YOLO fine-tune (GPU)

Fine-tunes **YOLO26s** (COCO-pretrained, ~10M params) on the CivicLoop vision set built by `python -m vision.dataset` (`data/vision/yolo/`): 4 box classes (pothole, garbage, damaged_sign, damaged_pole). The "other" and "none" photos have no boxes: they are background images that teach the model to stay quiet.

**Input:** `yolo.zip` (a zip of `data/vision/yolo/`), or the same folder already extracted. The first cell checks it is byte-for-byte the local build (manifest hash + image counts).
- **Kaggle:** add the dataset as input, GPU T4, Internet ON, Run All (~45 min).
- **Colab:** Runtime -> T4 GPU, upload `yolo.zip` to `/content`, Run All. The last cell downloads the result.

**Output:** `yolo_v1.zip` (best.pt, meta.json, results.csv, plots). Unzip it into `models/vision/yolo_v1/`, then run `python -m vision.evaluate --model yolo` locally.

The mAP printed here is Ultralytics' own (101-point, val split). The number to quote is the local one: VOC all-point, the same code that scored the CLIP baseline, on the untouched test split.

In [ ]:
# Pinned to the laptop's version so best.pt loads there unchanged.
%pip install -q "ultralytics==8.4.175"

In [ ]:
import hashlib
import json
import os
import shutil
import time
import zipfile
from pathlib import Path

import torch

SEED = 42  # = config.SEED
ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = not ON_KAGGLE and Path("/content").exists()
SMOKE = os.environ.get("CIVICLOOP_SMOKE") == "1"  # tiny CPU run: checks the code path only
GPU = torch.cuda.is_available()
WORK = Path(os.environ.get("CIVICLOOP_OUT")
            or ("/kaggle/working" if ON_KAGGLE else "/content" if ON_COLAB else "."))

BASE_MODEL = "yolo26s.pt"  # "s" = small: big enough to learn, fast enough to serve on CPU
# A short run: at most 60 epochs, and early stopping ends it once val mAP has not improved
# for 15 epochs. Ultralytics' default augmentation (mosaic, flips, colour jitter) is kept.
TRAIN = (dict(epochs=1, fraction=0.03, imgsz=320, batch=8, workers=0) if SMOKE
         else dict(epochs=60, patience=15, imgsz=640, batch=16, workers=4))

# The local build: data/vision/yolo/manifest.csv. Another hash means another dataset.
MANIFEST_SHA256 = "a0fea35dde65c54491b829e477655ed4bc56eab1f45e3d18e427984204a61f00"
EXPECTED_IMAGES = {"train": 2341, "val": 502, "test": 502}
print(f"GPU={torch.cuda.get_device_name(0) if GPU else None}  SMOKE={SMOKE}  WORK={WORK}")

In [ ]:
def find_data(roots=("/kaggle/input", "/content")):
    """The folder holding data.yaml: CIVICLOOP_DATA if set, else an extracted yolo/ folder or a
    yolo*.zip (extracted into WORK) at most 5 levels under an input root."""
    if os.environ.get("CIVICLOOP_DATA"):
        return Path(os.environ["CIVICLOOP_DATA"])
    roots = [Path(r) for r in roots if Path(r).exists()]
    levels = ["*/" * k for k in range(6)]  # Kaggle: /kaggle/input/datasets/<owner>/<slug>/yolo
    for root in roots:
        for yaml in sorted(p for lv in levels for p in root.glob(lv + "data.yaml")):
            if (yaml.parent / "manifest.csv").exists():
                return yaml.parent
    for root in roots:
        for z in sorted(p for lv in levels for p in root.glob(lv + "yolo*.zip")):
            with zipfile.ZipFile(z) as f:
                f.extractall(WORK / "_data")
            return next((WORK / "_data").rglob("data.yaml")).parent
    found = sorted(str(p) for r in roots for lv in levels[:3] for p in r.glob(lv + "*"))[:40]
    raise FileNotFoundError("No data.yaml or yolo.zip found. Kaggle: Add Input -> the dataset. "
                            "Colab: upload yolo.zip to /content. Found:\n" + "\n".join(found))


DATA = find_data()
digest = hashlib.sha256((DATA / "manifest.csv").read_bytes()).hexdigest()
assert digest == MANIFEST_SHA256, f"manifest.csv is {digest}, the local build is {MANIFEST_SHA256}"
counts = {s: len(list((DATA / "images" / s).glob("*.jpg"))) for s in EXPECTED_IMAGES}
assert counts == EXPECTED_IMAGES, f"image counts {counts} != {EXPECTED_IMAGES}"
print(DATA, counts, (DATA / "data.yaml").read_text(), sep="\n")

In [ ]:
from ultralytics import YOLO
from ultralytics import __version__ as ULTRALYTICS_VERSION

DEVICE = 0 if GPU else "cpu"  # one GPU: Kaggle's T4 x2 would otherwise run DDP
t0 = time.time()
model = YOLO(BASE_MODEL)  # COCO weights; the 80-class head is replaced by our 4 classes
model.train(data=str(DATA / "data.yaml"), seed=SEED, deterministic=True, device=DEVICE,
            project=str(WORK / "runs"), name="yolo_v1", exist_ok=True, **TRAIN)
train_minutes = (time.time() - t0) / 60
RUN = Path(model.trainer.save_dir)
print(f"trained in {train_minutes:.1f} min -> {RUN}")

In [ ]:
# Ultralytics' own score on val, for reference only. Test stays untouched until the local eval.
best = YOLO(RUN / "weights" / "best.pt")
m = best.val(data=str(DATA / "data.yaml"), split="val", imgsz=TRAIN["imgsz"],
             batch=TRAIN["batch"], device=DEVICE, plots=False, project=str(WORK / "runs"),
             name="val", exist_ok=True)
names = [best.names[i] for i in sorted(best.names)]
ultra_val = {"map50": float(m.box.map50), "map50_95": float(m.box.map),
             "ap50": {names[c]: float(a) for c, a in zip(m.box.ap_class_index, m.box.ap50)}}
print(json.dumps(ultra_val, indent=1))

In [ ]:
OUT = WORK / "yolo_v1"
OUT.mkdir(exist_ok=True)
shutil.copy(RUN / "weights" / "best.pt", OUT / "best.pt")
for f in ["args.yaml", "results.csv", "results.png", "confusion_matrix.png", "BoxPR_curve.png"]:
    if (RUN / f).exists():
        shutil.copy(RUN / f, OUT / f)
meta = {
    "step": "5.3", "base_model": BASE_MODEL, "ultralytics": ULTRALYTICS_VERSION,
    "torch": torch.__version__, "device": torch.cuda.get_device_name(0) if GPU else "cpu",
    "seed": SEED, "train_args": TRAIN, "train_minutes": round(train_minutes, 1),
    "epochs_run": sum(1 for _ in open(RUN / "results.csv")) - 1,
    "manifest_sha256": MANIFEST_SHA256, "ultralytics_val_101pt": ultra_val, "smoke": SMOKE,
}
(OUT / "meta.json").write_text(json.dumps(meta, indent=2))
shutil.make_archive(str(WORK / "yolo_v1"), "zip", OUT)

# Keep the output small (Kaggle saves everything in /kaggle/working): only yolo_v1 remains.
# weights/ holds yolo26n.pt, which Ultralytics' AMP check downloads.
for p in (WORK / "runs", WORK / "_data", WORK / "weights"):
    shutil.rmtree(p, ignore_errors=True)
(WORK / BASE_MODEL).unlink(missing_ok=True)
print(json.dumps(meta, indent=1))
print(sorted(str(p.relative_to(WORK)) for p in WORK.rglob("*")))
if ON_COLAB:
    from google.colab import files
    files.download(str(WORK / "yolo_v1.zip"))